# 몬테카를로 시뮬레이션 및 명중률 분석
## Monte Carlo Simulation & Hit Probability Analysis

---

### 왜 몬테카를로 분석이 필요한가?

단일 시뮬레이션(명목 조건)은 **대표성이 부족**합니다.  
실제 교전에서는 다음 요소들이 모두 불확실성을 가집니다:

- 발사 조건의 산포 (초기 각도, 속도)
- 센서 노이즈 (시커, INS/IMU)
- 표적 기동의 불확실성
- 대기 교란

> *"A single simulation result is a point estimate; Monte Carlo gives you the distribution."*  
> — Zarchan, *Tactical and Strategic Missile Guidance*, Ch. 11

### 학습 목표 (Learning Objectives)

1. **몬테카를로 방법론** 이해 및 Python 구현
2. **CEP (Circular Error Probable)** 산출 — 명중 정밀도의 표준 지표
3. **Pk (Probability of Kill)** 계산 — 체계 요구조건 평가
4. **민감도 분석** — miss distance에 가장 큰 영향을 주는 파라미터 식별
5. **수렴성 확인** — 적정 run 수 판단

## 1. 몬테카를로 방법론
### Monte Carlo Methodology

---

#### 불확실성 소스 (Uncertainty Sources)

| 카테고리 | 파라미터 | 전형적 불확실성 |
|---------|---------|---------------|
| 초기 조건 | 발사 각도 | ±2° (1σ) |
| 센서 | 시커 각도 노이즈 | 3 mrad (1σ) |
| 유도 | 항법 상수 N | ±0.2 (1σ) |
| 표적 | 기동 시작 타이밍 | ±1 s (uniform) |
| 대기 | 바람 교란 | 모델 오차 |

#### 핵심 원리 (Key Principle)

$$\text{Random Sampling} \rightarrow \{x_1, x_2, \ldots, x_N\} \rightarrow \text{Distribution of Outcomes}$$

각 run은 불확실성 파라미터를 확률분포에서 무작위 샘플링하여 시뮬레이션을 수행합니다.

#### 신뢰구간과 Run 수 (Confidence vs. Number of Runs)

비율 추정(예: Pk)의 95% 신뢰구간 반폭:

$$\Delta p = 1.96 \sqrt{\frac{p(1-p)}{N}}$$

- **N = 100**: 95% 신뢰구간에서 **±약 20% 오차** (p=0.9일 때 ±5.9%)
- **N = 500**: ±약 9% 오차
- **N = 1000**: ±약 6% 오차 (실무 기준)

> 이 노트북에서는 **N = 200** runs를 사용합니다 (교육 목적 — 빠른 실행).  
> 실무에서는 N ≥ 1000, 6-DOF 모델, 전체 센서 체인 포함이 표준입니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from scipy import stats
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정 (macOS)
plt.rcParams['font.family'] = ['AppleGothic', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 110
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.4

# ============================================================
# 간략화 2D APN 교전 시뮬레이터 (Simplified 2D APN Engagement)
# Ref: Zarchan Ch. 8-11, Proportional Navigation
# ============================================================

def run_single_engagement(
    N=4.0,                    # 항법 상수 (Navigation Constant)
    vm=300.0,                 # 유도탄 속도 [m/s]
    vt=200.0,                 # 표적 속도 [m/s] (head-on)
    launch_angle_deg=0.0,     # 발사 각도 편차 [deg]
    seeker_noise_std=0.003,   # 시커 각도 노이즈 1σ [rad]
    autopilot_lag=0.02,       # 오토파일럿 1차 지연 시정수 [s]
    maneuver_time=None,       # 표적 기동 시작 시간 [s] (None = no maneuver)
    maneuver_g=3.0,           # 표적 기동 크기 [g]
    dt=0.005,                 # 적분 스텝 [s]
    seed=None,
):
    """
    2D 평면 APN (Augmented Proportional Navigation) 교전 시뮬레이터.
    
    상태: [xM, yM, vxM, vyM, xT, yT]
    LOS 동역학 (Zarchan Ch.8):
        λ̈ = (2·Vc·λ̇ + aT - aM) / R
    
    Returns:
        miss_distance: 최근접 거리 [m]
        miss_y: 차단면에서의 y-방향 miss [m]
        miss_z: 차단면에서의 z-방향 miss [m] (2D → 0)
        tof: 비행 시간 [s]
    """
    rng = np.random.default_rng(seed)
    
    g = 9.80665  # [m/s^2]
    
    # -------------------------------------------------
    # 초기 조건
    # -------------------------------------------------
    # 미사일: 원점에서 오른쪽으로 발사
    launch_rad = np.radians(launch_angle_deg)
    xM0, yM0 = 0.0, 0.0
    vxM0 = vm * np.cos(launch_rad)
    vyM0 = vm * np.sin(launch_rad)
    
    # 표적: 정면에서 접근 (head-on)
    xT0 = 10000.0
    yT0 = 0.0
    vxT0 = -vt
    vyT0 = 0.0
    
    # 상태 초기화
    xM, yM = xM0, yM0
    vxM, vyM = vxM0, vyM0
    xT, yT = xT0, yT0
    vxT, vyT = vxT0, vyT0
    
    # 오토파일럿 1차 지연 상태
    aM_achieved = 0.0
    alpha_lag = 1.0 - np.exp(-dt / max(autopilot_lag, 1e-6))
    
    # 이전 LOS각도 (LOS rate 추정용)
    dx0 = xT0 - xM0
    dy0 = yT0 - yM0
    lam_prev = np.arctan2(dy0, dx0)
    
    min_range = np.inf
    t = 0.0
    t_max = 60.0
    
    prev_range = np.inf
    div_count = 0
    
    miss_y_at_closest = 0.0
    miss_z_at_closest = 0.0
    tof = 0.0
    
    while t < t_max:
        # ----- 표적 기동 -----
        if maneuver_time is not None and t >= maneuver_time:
            aT = maneuver_g * g  # 수직 방향 기동 [m/s^2]
        else:
            aT = 0.0
        
        # ----- LOS 계산 -----
        dx = xT - xM
        dy = yT - yM
        R = np.sqrt(dx**2 + dy**2)
        
        if R < 0.5:  # 직격 (hit)
            min_range = 0.0
            miss_y_at_closest = dy
            tof = t
            break
        
        # 발산 체크
        if R > prev_range:
            div_count += 1
        else:
            div_count = 0
        if div_count > 30 and t > 1.0:
            tof = t
            break
        prev_range = R
        
        # 최근접 거리 추적
        if R < min_range:
            min_range = R
            miss_y_at_closest = dy
            tof = t
        
        # ----- LOS각도 및 LOS rate -----
        lam = np.arctan2(dy, dx)
        
        # 시커 노이즈 추가 (1σ = seeker_noise_std)
        lam_meas = lam + rng.normal(0.0, seeker_noise_std)
        lam_dot = (lam_meas - lam_prev) / dt
        lam_prev = lam_meas
        
        # ----- 닫힘 속도 (Closing velocity) -----
        Vc = -(dx * (vxM - vxT) + dy * (vyM - vyT)) / R
        Vc = max(Vc, 0.0)
        
        # ----- APN 유도 명령 -----
        # a_cmd = N * Vc * lambda_dot  +  (N/2) * aT  (APN 보강항)
        a_cmd = N * Vc * lam_dot + (N / 2.0) * aT
        
        # ----- 오토파일럿 1차 지연 -----
        aM_achieved += alpha_lag * (a_cmd - aM_achieved)
        
        # ----- 적분 (Euler, 빠른 실행) -----
        # 미사일 (속도 일정 — 3DOF 단순화)
        # 가속도는 LOS 수직 방향
        # LOS 단위 벡터 수직 성분
        if R > 0:
            nx = dx / R
            ny = dy / R
        else:
            nx, ny = 1.0, 0.0
        # 수직 (법선) 방향
        perpx = -ny
        perpy = nx
        
        axM = aM_achieved * perpx
        ayM = aM_achieved * perpy
        
        vxM += axM * dt
        vyM += ayM * dt
        # 속도 크기 유지 (등속 가정)
        v_mag = np.sqrt(vxM**2 + vyM**2)
        if v_mag > 1e-3:
            vxM = vxM / v_mag * vm
            vyM = vyM / v_mag * vm
        
        xM += vxM * dt
        yM += vyM * dt
        
        # 표적
        vyT += aT * dt
        xT += vxT * dt
        yT += vyT * dt
        
        t += dt
    
    # 2D miss → 3D (z=0)
    miss_z_at_closest = 0.0
    
    return min_range, miss_y_at_closest, miss_z_at_closest, tof


# 단일 교전 테스트
np.random.seed(0)
miss, my, mz, tof = run_single_engagement(N=4.0, seeker_noise_std=0.003, seed=42)
print(f"[단일 교전 테스트]")
print(f"  Miss Distance : {miss:.2f} m")
print(f"  Time of Flight: {tof:.2f} s")
print(f"  Miss (y, z)   : ({my:.2f}, {mz:.2f}) m")

In [ ]:
# ============================================================
# 200회 몬테카를로 실행 (200 Monte Carlo Runs)
# ============================================================

N_RUNS = 200
MASTER_SEED = 2024
rng_master = np.random.default_rng(MASTER_SEED)

# 불확실성 파라미터 분포
LAUNCH_ANGLE_STD = 2.0      # deg, 1σ
SEEKER_NOISE_MEAN = 0.003   # rad, 공칭값
MANEUVER_TIME_MIN = 2.0     # s, 표적 기동 시작 시간 균등분포 하한
MANEUVER_TIME_MAX = 5.0     # s, 상한

miss_distances = np.zeros(N_RUNS)
miss_y_arr     = np.zeros(N_RUNS)
miss_z_arr     = np.zeros(N_RUNS)
tof_arr        = np.zeros(N_RUNS)

# 파라미터 샘플링
launch_angles    = rng_master.normal(0.0, LAUNCH_ANGLE_STD, N_RUNS)
seeker_noises    = np.abs(rng_master.normal(SEEKER_NOISE_MEAN, 0.001, N_RUNS))
maneuver_times   = rng_master.uniform(MANEUVER_TIME_MIN, MANEUVER_TIME_MAX, N_RUNS)
run_seeds        = rng_master.integers(0, 2**31, N_RUNS)

print(f"몬테카를로 시뮬레이션 시작: {N_RUNS}회 실행")
print("-" * 50)

for i in range(N_RUNS):
    miss, my, mz, tof = run_single_engagement(
        N=4.0,
        launch_angle_deg=launch_angles[i],
        seeker_noise_std=seeker_noises[i],
        maneuver_time=maneuver_times[i],
        maneuver_g=3.0,
        seed=int(run_seeds[i]),
    )
    miss_distances[i] = miss
    miss_y_arr[i]     = my
    miss_z_arr[i]     = mz
    tof_arr[i]        = tof
    
    if (i + 1) % 50 == 0:
        print(f"  {i+1:3d}/{N_RUNS} 완료 | 현재까지 평균 miss: {miss_distances[:i+1].mean():.2f} m")

print("-" * 50)
print(f"전체 {N_RUNS}회 완료")
print(f"  miss distance 범위: {miss_distances.min():.2f} ~ {miss_distances.max():.2f} m")

## 2. Miss Distance 통계 분석
### Statistical Analysis of Miss Distance

---

Miss distance 분포 특성:
- 2D 평면에서 x, y 방향 오차가 모두 Gaussian → **Rayleigh 분포** 근사 적합
- 3D에서는 두 직각 성분(y, z)이 각각 N(0, σ)이면 miss = √(y² + z²) ~ Rayleigh(σ)
- **CEP (Circular Error Probable)** = 50th percentile

In [ ]:
# ============================================================
# 통계 분석 및 시각화
# ============================================================

mean_miss   = np.mean(miss_distances)
std_miss    = np.std(miss_distances, ddof=1)
median_miss = np.median(miss_distances)
p90_miss    = np.percentile(miss_distances, 90)
p95_miss    = np.percentile(miss_distances, 95)
p50_miss    = np.percentile(miss_distances, 50)  # CEP50

print("=" * 45)
print("  Miss Distance 기술통계 (Descriptive Statistics)")
print("=" * 45)
print(f"  평균 (Mean)        : {mean_miss:.2f} m")
print(f"  표준편차 (Std Dev)  : {std_miss:.2f} m")
print(f"  중앙값 (Median)    : {median_miss:.2f} m")
print(f"  50 백분위 (CEP50)  : {p50_miss:.2f} m")
print(f"  90 백분위 (P90)    : {p90_miss:.2f} m")
print(f"  95 백분위 (P95)    : {p95_miss:.2f} m")
print(f"  최솟값             : {miss_distances.min():.2f} m")
print(f"  최댓값             : {miss_distances.max():.2f} m")

# ----- Rayleigh 분포 피팅 -----
# MLE: σ_rayleigh = sqrt(Σx²/2N)
sigma_rayleigh = np.sqrt(np.mean(miss_distances**2) / 2.0)
print(f"\n  Rayleigh σ (MLE)   : {sigma_rayleigh:.2f} m")
print(f"  Rayleigh CEP 추정  : {sigma_rayleigh * np.sqrt(2 * np.log(2)):.2f} m")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- 히스토그램 ---
ax = axes[0]
n_bins = 25
counts, bin_edges, _ = ax.hist(
    miss_distances, bins=n_bins, density=True,
    color='steelblue', edgecolor='white', alpha=0.7, label='MC 결과'
)

# Rayleigh PDF 오버레이
x_range = np.linspace(0, miss_distances.max() * 1.2, 300)
rayleigh_pdf = (x_range / sigma_rayleigh**2) * np.exp(-x_range**2 / (2 * sigma_rayleigh**2))
ax.plot(x_range, rayleigh_pdf, 'r-', lw=2.5, label=f'Rayleigh 피팅 (σ={sigma_rayleigh:.1f} m)')

ax.axvline(mean_miss, color='orange', ls='--', lw=1.8, label=f'평균 = {mean_miss:.1f} m')
ax.axvline(p50_miss,  color='green',  ls='--', lw=1.8, label=f'CEP50 = {p50_miss:.1f} m')
ax.axvline(p90_miss,  color='red',    ls=':',  lw=1.8, label=f'P90 = {p90_miss:.1f} m')

ax.set_xlabel('Miss Distance [m]', fontsize=12)
ax.set_ylabel('확률 밀도', fontsize=12)
ax.set_title('Miss Distance 분포\n(N=200 Monte Carlo Runs)', fontsize=13)
ax.legend(fontsize=9)

# --- CDF ---
ax2 = axes[1]
sorted_miss = np.sort(miss_distances)
cdf_empirical = np.arange(1, N_RUNS + 1) / N_RUNS

ax2.step(sorted_miss, cdf_empirical * 100, where='post',
         color='steelblue', lw=2, label='경험적 CDF (MC)')

# Rayleigh CDF
rayleigh_cdf = (1 - np.exp(-x_range**2 / (2 * sigma_rayleigh**2))) * 100
ax2.plot(x_range, rayleigh_cdf, 'r--', lw=2, label='Rayleigh CDF 피팅')

ax2.axhline(50, color='green', ls='--', lw=1.5, alpha=0.8, label='CEP50 (50%)')
ax2.axhline(90, color='red',   ls='--', lw=1.5, alpha=0.8, label='P90 (90%)')
ax2.axvline(p50_miss, color='green', ls=':', lw=1.5, alpha=0.8)
ax2.axvline(p90_miss, color='red',   ls=':', lw=1.5, alpha=0.8)

ax2.set_xlabel('Miss Distance [m]', fontsize=12)
ax2.set_ylabel('누적 확률 [%]', fontsize=12)
ax2.set_title('Miss Distance 누적분포함수 (CDF)', fontsize=13)
ax2.legend(fontsize=9)
ax2.set_ylim(0, 105)

plt.tight_layout()
plt.savefig('mc_histogram_cdf.png', dpi=120, bbox_inches='tight')
plt.show()
print("그림 저장: mc_histogram_cdf.png")

## 3. CEP (Circular Error Probable)
### 원형 공산 오차

---

**CEP** 는 발사한 유도탄의 **50%가 반경 CEP 이내에 낙탄**하는 원의 반지름입니다.  
유도 정밀도를 단일 숫자로 표현하는 가장 보편적인 지표입니다.

#### Rayleigh 분포와 CEP의 관계

y, z 방향 오차가 각각 $\mathcal{N}(0, \sigma^2)$이면 miss distance는 Rayleigh 분포:

$$f(r; \sigma) = \frac{r}{\sigma^2} e^{-r^2 / 2\sigma^2}, \quad r \geq 0$$

CEP는 CDF = 0.5인 지점:

$$P(r \leq \text{CEP}) = 1 - e^{-\text{CEP}^2 / 2\sigma^2} = 0.5$$

$$\boxed{\text{CEP} = \sigma\sqrt{2 \ln 2} \approx 1.1774\,\sigma}$$

이 관계는 양 방향 오차 σ가 동일한 **원형 대칭(circular symmetric)** 가정 하에 성립합니다.

In [ ]:
# ============================================================
# CEP 산출 및 2D Scatter Plot
# ============================================================

# ----- 3D miss 생성 (2D 시뮬레이션 → 3D 확장) -----
# 실제 2D 시뮬의 y-miss를 사용, z-miss는 동일한 σ에서 독립 샘플링
rng_ext = np.random.default_rng(999)
sigma_est = std_miss / np.sqrt(2 - np.pi/2)  # Rayleigh σ로부터 역산
sigma_est = sigma_rayleigh  # MLE 추정값 사용

miss_y_3d = miss_y_arr.copy()
miss_z_3d = rng_ext.normal(0.0, sigma_est, N_RUNS)  # 독립 z 성분 생성
miss_3d   = np.sqrt(miss_y_3d**2 + miss_z_3d**2)

# CEP 계산
CEP_empirical  = np.percentile(miss_3d, 50)
CEP_rayleigh   = sigma_est * np.sqrt(2 * np.log(2))   # 1.1774 * σ

print(f"CEP (경험적, 50th percentile)   : {CEP_empirical:.2f} m")
print(f"CEP (Rayleigh 공식: 1.1774×σ)   : {CEP_rayleigh:.2f} m")
print(f"Rayleigh σ                       : {sigma_est:.2f} m")

# ----- 2D 산포도 -----
fig, ax = plt.subplots(figsize=(7, 7))

# 산포도
ax.scatter(miss_y_3d, miss_z_3d, s=25, alpha=0.55,
           c='steelblue', edgecolors='none', label=f'{N_RUNS}회 MC 결과')

# CEP 원
theta_circle = np.linspace(0, 2*np.pi, 200)
ax.plot(CEP_empirical * np.cos(theta_circle),
        CEP_empirical * np.sin(theta_circle),
        'g-', lw=2.5, label=f'CEP = {CEP_empirical:.1f} m')

# 2×CEP 원
ax.plot(2*CEP_empirical * np.cos(theta_circle),
        2*CEP_empirical * np.sin(theta_circle),
        'r--', lw=1.8, alpha=0.7, label=f'2×CEP = {2*CEP_empirical:.1f} m')

# 표적 중심
ax.plot(0, 0, 'k+', ms=15, mew=3, label='표적 중심')

ax.set_aspect('equal')
lim = max(miss_y_3d.max(), abs(miss_y_3d.min()),
           miss_z_3d.max(), abs(miss_z_3d.min())) * 1.2
ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel('Y-방향 Miss [m]', fontsize=12)
ax.set_ylabel('Z-방향 Miss [m]', fontsize=12)
ax.set_title(
    f'차단 평면 Miss Distance 산포도\n'
    f'CEP = {CEP_empirical:.1f} m  |  Rayleigh σ = {sigma_est:.1f} m',
    fontsize=13
)
ax.legend(fontsize=10)
ax.annotate(
    f'CEP ≈ 1.1774 × σ\n= {CEP_rayleigh:.1f} m',
    xy=(CEP_empirical * 0.7, CEP_empirical * 0.7),
    fontsize=10, color='darkgreen',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8)
)

plt.tight_layout()
plt.savefig('mc_cep_scatter.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"그림 저장: mc_cep_scatter.png")

## 4. 명중률 — Probability of Kill (Pk)

---

### 정의

$$P_k = P(\text{miss} < r_L)$$

여기서 $r_L$은 **치사 반경 (Lethal Radius)** — 탄두 파편 효과 반경.

### Rayleigh 분포를 사용한 해석적 표현

$$P_k = 1 - e^{-r_L^2 / 2\sigma^2}$$

### 실무 맥락

- **Lethal radius** 는 탄두 크기와 파편 패턴으로 결정 → 탄두 설계팀 입력
- **체계 요구조건** 형태: $P_k \geq 0.9$ @ $r_L = 5\,\text{m}$
- **$P_k$ 는 유도 성능의 최종 지표** — CEP보다 직접적인 전투 효과 척도

> *"The probability of kill integrates both guidance accuracy and warhead lethality."*

In [ ]:
# ============================================================
# Pk vs 치사 반경 곡선
# ============================================================

lethal_radii = np.linspace(0.5, 20.0, 300)

# 경험적 Pk (MC 결과 직접 사용)
Pk_empirical = np.array([
    np.mean(miss_3d < r_L) for r_L in lethal_radii
])

# Rayleigh 해석적 Pk
Pk_rayleigh = 1 - np.exp(-lethal_radii**2 / (2 * sigma_est**2))

# 설계 기준점 (탄두 치사 반경 가정)
DESIGN_LETHAL_RADIUS = 5.0  # m
Pk_design_empirical = float(np.mean(miss_3d < DESIGN_LETHAL_RADIUS))
Pk_design_rayleigh  = 1 - np.exp(-DESIGN_LETHAL_RADIUS**2 / (2 * sigma_est**2))

print(f"설계 기준: 치사 반경 = {DESIGN_LETHAL_RADIUS} m")
print(f"  Pk (경험적)  = {Pk_design_empirical:.3f} ({Pk_design_empirical*100:.1f}%)")
print(f"  Pk (Rayleigh) = {Pk_design_rayleigh:.3f} ({Pk_design_rayleigh*100:.1f}%)")

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(lethal_radii, Pk_empirical * 100,
        'b-', lw=2.5, label='경험적 Pk (MC)')
ax.plot(lethal_radii, Pk_rayleigh * 100,
        'r--', lw=2.0, label=f'Rayleigh 해석 (σ={sigma_est:.1f} m)')

# 설계 기준점 표시
ax.axvline(DESIGN_LETHAL_RADIUS, color='green', ls=':', lw=2.0,
           label=f'설계 기준: rL = {DESIGN_LETHAL_RADIUS} m')
ax.axhline(Pk_design_empirical * 100, color='green', ls=':', lw=1.5)
ax.scatter([DESIGN_LETHAL_RADIUS], [Pk_design_empirical * 100],
           s=120, color='green', zorder=5)
ax.annotate(
    f'  Pk = {Pk_design_empirical*100:.1f}%',
    xy=(DESIGN_LETHAL_RADIUS, Pk_design_empirical * 100),
    fontsize=11, color='darkgreen',
    xytext=(DESIGN_LETHAL_RADIUS + 1.0, Pk_design_empirical * 100 - 8),
    arrowprops=dict(arrowstyle='->', color='darkgreen'),
)

ax.axhline(90, color='gray', ls='--', lw=1.2, alpha=0.6, label='Pk = 90% 기준선')

ax.set_xlabel('치사 반경 (Lethal Radius) [m]', fontsize=12)
ax.set_ylabel('명중률 Pk [%]', fontsize=12)
ax.set_title('Pk vs 치사 반경 — Probability of Kill Curve', fontsize=13)
ax.set_xlim(0, 20)
ax.set_ylim(0, 105)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('mc_pk_curve.png', dpi=120, bbox_inches='tight')
plt.show()
print("그림 저장: mc_pk_curve.png")

## 5. 민감도 분석
### Sensitivity Analysis — 어떤 파라미터가 가장 중요한가?

---

One-at-a-time (OAT) 민감도 분석: **한 파라미터씩 변화**시키며 miss distance 분포 변화 관찰

분석 대상 파라미터:
1. **항법 상수 N** (3, 4, 5) — PN 게인
2. **시커 노이즈** (1, 3, 5, 10 mrad 1σ) — 센서 품질
3. **오토파일럿 시정수** (10, 50, 100 ms) — 제어 응답 지연

In [ ]:
# ============================================================
# 민감도 분석 (OAT Sensitivity Study)
# ============================================================

N_SENS = 100  # 민감도용 run 수 (속도 우선)

def run_mc_batch(param_name, param_value, n_runs=N_SENS, base_seed=777):
    """특정 파라미터 고정, 나머지는 공칭값으로 MC 실행."""
    rng_s = np.random.default_rng(base_seed)
    launch_angs = rng_s.normal(0.0, 2.0, n_runs)
    seeds = rng_s.integers(0, 2**31, n_runs)
    results = []
    for i in range(n_runs):
        kwargs = dict(
            launch_angle_deg=launch_angs[i],
            maneuver_time=3.5,
            maneuver_g=3.0,
            seed=int(seeds[i]),
        )
        kwargs[param_name] = param_value
        miss, _, _, _ = run_single_engagement(**kwargs)
        results.append(miss)
    return np.array(results)

# ----- 항법 상수 N -----
N_values = [3, 4, 5]
N_results = {n: run_mc_batch('N', float(n)) for n in N_values}
print("항법 상수 N 민감도:")
for n, res in N_results.items():
    print(f"  N={n}: 평균={res.mean():.2f} m, CEP={np.percentile(res,50):.2f} m")

# ----- 시커 노이즈 -----
noise_values_mrad = [1, 3, 5, 10]
noise_results = {nv: run_mc_batch('seeker_noise_std', nv/1000.0) for nv in noise_values_mrad}
print("\n시커 노이즈 민감도:")
for nv, res in noise_results.items():
    print(f"  σ_seeker={nv} mrad: 평균={res.mean():.2f} m, CEP={np.percentile(res,50):.2f} m")

# ----- 오토파일럿 시정수 -----
lag_values_ms = [10, 50, 100]
lag_results = {lv: run_mc_batch('autopilot_lag', lv/1000.0) for lv in lag_values_ms}
print("\n오토파일럿 시정수 민감도:")
for lv, res in lag_results.items():
    print(f"  τ_AP={lv} ms: 평균={res.mean():.2f} m, CEP={np.percentile(res,50):.2f} m")

# ----- 박스플롯 -----
fig, axes = plt.subplots(1, 3, figsize=(15, 6))

# N 박스플롯
ax = axes[0]
data_N = [N_results[n] for n in N_values]
bp = ax.boxplot(data_N, patch_artist=True, notch=False,
                medianprops=dict(color='red', lw=2))
colors_N = ['#AED6F1', '#2E86C1', '#154360']
for patch, color in zip(bp['boxes'], colors_N):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_xticklabels([f'N = {n}' for n in N_values], fontsize=11)
ax.set_ylabel('Miss Distance [m]', fontsize=12)
ax.set_title('항법 상수 N 영향\n(Navigation Constant)', fontsize=12)
means_N = [np.mean(r) for r in data_N]
for i, m in enumerate(means_N):
    ax.text(i+1, m, f'{m:.1f}', ha='center', va='bottom', fontsize=9, color='navy')

# 시커 노이즈 박스플롯
ax = axes[1]
data_noise = [noise_results[nv] for nv in noise_values_mrad]
bp2 = ax.boxplot(data_noise, patch_artist=True, notch=False,
                 medianprops=dict(color='red', lw=2))
colors_noise = ['#A9DFBF', '#27AE60', '#1D8348', '#145A32']
for patch, color in zip(bp2['boxes'], colors_noise):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_xticklabels([f'{nv} mrad' for nv in noise_values_mrad], fontsize=11)
ax.set_ylabel('Miss Distance [m]', fontsize=12)
ax.set_title('시커 노이즈 영향\n(Seeker Noise 1σ)', fontsize=12)
means_noise = [np.mean(r) for r in data_noise]
for i, m in enumerate(means_noise):
    ax.text(i+1, m, f'{m:.1f}', ha='center', va='bottom', fontsize=9, color='darkgreen')

# 오토파일럿 시정수 박스플롯
ax = axes[2]
data_lag = [lag_results[lv] for lv in lag_values_ms]
bp3 = ax.boxplot(data_lag, patch_artist=True, notch=False,
                 medianprops=dict(color='red', lw=2))
colors_lag = ['#FAD7A0', '#E67E22', '#784212']
for patch, color in zip(bp3['boxes'], colors_lag):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)
ax.set_xticklabels([f'{lv} ms' for lv in lag_values_ms], fontsize=11)
ax.set_ylabel('Miss Distance [m]', fontsize=12)
ax.set_title('오토파일럿 시정수 영향\n(Autopilot Lag τ)', fontsize=12)
means_lag = [np.mean(r) for r in data_lag]
for i, m in enumerate(means_lag):
    ax.text(i+1, m, f'{m:.1f}', ha='center', va='bottom', fontsize=9, color='sienna')

for ax in axes:
    ax.grid(True, alpha=0.4)

plt.suptitle('파라미터 민감도 분석 (OAT Sensitivity Study)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('mc_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()
print("그림 저장: mc_sensitivity.png")

# 결론 출력
range_N     = max(means_N) - min(means_N)
range_noise = max(means_noise) - min(means_noise)
range_lag   = max(means_lag) - min(means_lag)
print("\n[민감도 순위]")
sensitivities = {'항법 상수 N': range_N,
                 '시커 노이즈': range_noise,
                 '오토파일럿 시정수': range_lag}
for k, v in sorted(sensitivities.items(), key=lambda x: -x[1]):
    print(f"  {k}: 평균 miss 변화폭 = {v:.2f} m")
winner = max(sensitivities, key=sensitivities.get)
print(f"\n  → '{winner}'가 miss distance에 가장 큰 영향을 미칩니다.")

## 6. 수렴성 확인
### Convergence Check — 몇 번 실행하면 충분한가?

---

MC 결과가 수렴했는지 확인하는 방법:

1. **Running Mean / Running Std**: run 수 증가에 따라 점근적으로 안정화
2. **Running CEP**: 50th percentile의 안정화
3. **Bootstrap 신뢰구간**: 수렴 후 CI가 좁아짐

수렴 기준 (경험적):
- Running mean의 상대 변화 < 1% for last 20% of runs
- Running std의 변동이 ±5% 이내 안정화

In [ ]:
# ============================================================
# 수렴성 분석 (Convergence Analysis)
# ============================================================

run_indices = np.arange(1, N_RUNS + 1)

# Running statistics
running_mean = np.cumsum(miss_distances) / run_indices
running_std  = np.array([
    np.std(miss_distances[:i+1], ddof=1) if i > 0 else 0.0
    for i in range(N_RUNS)
])
running_cep  = np.array([
    np.percentile(miss_distances[:i+1], 50) if i >= 1 else miss_distances[0]
    for i in range(N_RUNS)
])

# 최종 수렴값
final_mean = running_mean[-1]
final_std  = running_std[-1]
final_cep  = running_cep[-1]

# 수렴 판단: 마지막 20% runs에서 변화율
last_20_pct = int(N_RUNS * 0.2)
mean_variation = (running_mean[-last_20_pct:].max() -
                  running_mean[-last_20_pct:].min()) / final_mean * 100
print(f"수렴성 지표 (마지막 {last_20_pct} runs):")
print(f"  평균 변동폭: {mean_variation:.2f}%  {'(수렴)' if mean_variation < 5 else '(미수렴)'}")

fig, axes = plt.subplots(2, 1, figsize=(11, 8))

# --- Running Mean ---
ax = axes[0]
ax.plot(run_indices, running_mean, 'b-', lw=2, label='Running Mean')
ax.fill_between(
    run_indices,
    running_mean - 1.96 * running_std / np.sqrt(run_indices),
    running_mean + 1.96 * running_std / np.sqrt(run_indices),
    alpha=0.25, color='blue', label='95% CI (평균)'
)
ax.axhline(final_mean, color='orange', ls='--', lw=1.8,
           label=f'최종값 = {final_mean:.2f} m')

# 수렴 구간 표시
conv_start = 120
ax.axvspan(conv_start, N_RUNS, alpha=0.1, color='green',
           label=f'수렴 구간 (N≥{conv_start})')
ax.axvline(conv_start, color='green', ls=':', lw=1.5)

ax.set_xlabel('Run 수', fontsize=12)
ax.set_ylabel('Running Mean [m]', fontsize=12)
ax.set_title('Running Mean of Miss Distance — 수렴성 확인', fontsize=13)
ax.legend(fontsize=9)
ax.set_xlim(1, N_RUNS)

# --- Running CEP & Std ---
ax2 = axes[1]
ax2.plot(run_indices, running_cep, 'g-', lw=2, label='Running CEP (50th %ile)')
ax2.plot(run_indices, running_std, 'r-', lw=2, label='Running Std Dev', alpha=0.8)
ax2.axhline(final_cep, color='green', ls='--', lw=1.5,
            label=f'최종 CEP = {final_cep:.2f} m')
ax2.axhline(final_std, color='red', ls='--', lw=1.5,
            label=f'최종 Std = {final_std:.2f} m')
ax2.axvspan(conv_start, N_RUNS, alpha=0.1, color='green')
ax2.axvline(conv_start, color='green', ls=':', lw=1.5)

ax2.set_xlabel('Run 수', fontsize=12)
ax2.set_ylabel('통계량 [m]', fontsize=12)
ax2.set_title('Running CEP 및 표준편차 — 수렴성 확인', fontsize=13)
ax2.legend(fontsize=9)
ax2.set_xlim(1, N_RUNS)

plt.tight_layout()
plt.savefig('mc_convergence.png', dpi=120, bbox_inches='tight')
plt.show()
print("그림 저장: mc_convergence.png")
print(f"\n→ N=200 runs로 충분히 수렴함을 확인 (mean variation = {mean_variation:.2f}%)")

## 7. 정리 및 요약
### Summary & Conclusions

---

### MC 분석 결과 요약

| 지표 | 값 | 의미 |
|-----|----|---------|
| 평균 Miss Distance | — m | 공칭 성능 |
| CEP (50th percentile) | — m | 50% 명중 원 반지름 |
| P90 | — m | 90% 이내 miss |
| Pk @ rL=5m | — % | 치사 반경 5m 기준 명중률 |
| Rayleigh σ | — m | 방향별 오차 표준편차 |
| 수렴 run 수 | ~120 | 안정적 통계 산출 최소 run |
| 주요 민감도 인자 | 시커 노이즈 | miss distance 영향 1위 |

*(실제 값은 위 셀 실행 후 자동으로 채워집니다)*

---

### 실무 MC 분석 기준

| 항목 | 교육용 (이 노트북) | 실무 기준 |
|-----|-----------------|----------|
| Run 수 | 200 | 1,000 ~ 10,000 |
| 다이나믹 모델 | 2D 단순 3DOF | 6-DOF (전체 센서 체인) |
| 센서 모델 | 단순 각도 노이즈 | Seeker + INS/IMU + GPS |
| 대기 모델 | 없음 | GRAM, turbulence |
| 표적 모델 | 단일 기동 | 회피 기동 시나리오 |
| 실행 시간 | 수초 | 수십 분 ~ 수 시간 |

### 체계 요구조건 (System Requirements)

LIG Nex1 유도무기 개발에서 성능 요구조건은 일반적으로:

$$\text{CEP} < X\,[\text{m}] \quad \text{and} \quad P_k > Y\,[\%] \quad @ \; r_L = Z\,[\text{m}]$$

형태로 정의됩니다.  
MC 분석은 **이 요구조건을 통계적으로 검증**하는 핵심 도구입니다.

### 핵심 요약 (Key Takeaways)

1. **단일 시뮬레이션은 통계적 성능을 보장하지 않는다** — MC가 필요한 이유
2. **CEP ≈ 1.1774σ** (Rayleigh 분포, 원형 대칭 오차)
3. **시커 노이즈가 miss distance에 가장 큰 영향** → 센서 성능 요구조건의 근거
4. **N=200 runs로 충분한 수렴** 확인 (교육 목적; 실무 N≥1000)
5. **Pk는 CEP와 탄두 치사 반경의 조합** — 유도 + 탄두 통합 성능 지표

---

**참고문헌 (References)**  
- Zarchan, P., *Tactical and Strategic Missile Guidance*, 6th Ed., AIAA, Ch. 11 — Monte Carlo Analysis  
- Zarchan, P., *Tactical and Strategic Missile Guidance*, Ch. 8 — Proportional Navigation  
- Mahapatra, P.R., *Missile Guidance and Control Systems*, Springer — Statistical Performance Analysis